# Reusable Template: Numerical Feature Transformations

**Purpose:** Drop-in starting point for any tabular project that needs centering, scaling, log transforms or binning before modelling.

**How to reuse**
1. Replace the data-loading cell with your own CSV / query.
2. Edit the `NUMERIC_COLS` list.
3. Run the diagnosis → transform → (optional) model-impact cells.
4. Keep the simulation cell if you want a quick sensitivity check.

**Dependencies:** pandas, numpy, matplotlib, scikit-learn


## Configuration

In [ ]:
# === EDIT THESE ===
DATA_PATH = 'data/starbucks_customers.csv'   # change to your file
TARGET_COL = 'spent'                         # column you will predict (optional)
NUMERIC_COLS = ['age', 'nearest_starbucks', 'spent']  # columns to transform
LOG_CANDIDATES = ['age', 'spent']            # columns that may benefit from log1p
N_BINS = 3
RANDOM_STATE = 42


## 1. Load & diagnose

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression

df = pd.read_csv(DATA_PATH)
print(df[NUMERIC_COLS].describe())
print('\nSkew:\n', df[NUMERIC_COLS].skew())

fig, axes = plt.subplots(1, len(NUMERIC_COLS), figsize=(4*len(NUMERIC_COLS), 3))
if len(NUMERIC_COLS) == 1: axes = [axes]
for ax, col in zip(axes, NUMERIC_COLS):
    ax.hist(df[col], bins=15, color='#42A5F5', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Original distributions'); plt.tight_layout(); plt.show()


## 2. Apply transforms

In [ ]:
# Centering
centered = df[NUMERIC_COLS] - df[NUMERIC_COLS].mean()

# Standardization
scaler = StandardScaler()
standardized = pd.DataFrame(
    scaler.fit_transform(df[NUMERIC_COLS]),
    columns=[c + '_std' for c in NUMERIC_COLS],
    index=df.index
)

# Min-Max
mm = MinMaxScaler()
minmaxed = pd.DataFrame(
    mm.fit_transform(df[NUMERIC_COLS]),
    columns=[c + '_mm' for c in NUMERIC_COLS],
    index=df.index
)

# Log1p on selected columns
logged = df[NUMERIC_COLS].copy()
for c in LOG_CANDIDATES:
    if c in logged.columns:
        logged[c] = np.log1p(logged[c].clip(lower=0))

# Binning example (first numeric column)
first_col = NUMERIC_COLS[0]
df[first_col + '_bin'] = pd.cut(df[first_col], bins=N_BINS)

print('Standardized means (should be ~0):\n', standardized.mean())
print('MinMax ranges:\n', minmaxed.agg(['min','max']))


## 3. Optional – quick model impact simulation

In [ ]:
if TARGET_COL and TARGET_COL in df.columns and len(NUMERIC_COLS) >= 2:
    from sklearn.preprocessing import StandardScaler
    y = df[TARGET_COL].values
    X = df[[c for c in NUMERIC_COLS if c != TARGET_COL]].values
    def r2(transform):
        scores = []
        for _ in range(40):
            idx = np.random.choice(len(y), size=len(y), replace=True)
            Xb, yb = X[idx], y[idx]
            if transform == 'std':
                Xb = StandardScaler().fit_transform(Xb)
            scores.append(LinearRegression().fit(Xb, yb).score(Xb, yb))
        return np.mean(scores)
    print('Mean bootstrap R² raw :', r2('raw'))
    print('Mean bootstrap R² std :', r2('std'))
else:
    print('Skipping model simulation (need TARGET_COL and ≥2 predictors)')


## 4. Notes for the next project
- Fit every scaler on the **training** fold only.
- For pipelines use `sklearn.pipeline.Pipeline([('scaler', StandardScaler()), ('model', ...)])`.
- When outliers are severe consider `RobustScaler` (median / IQR) instead of StandardScaler.
- Document which transform was chosen and why (audience + algorithm requirements).
